# DX 603: Project Milestone Two: Modeling and Feature Engineering

### Due: Sunday July 26 @ 11:59PM (with grace period of 2 hours & 1 minute)

### Overview

In Milestone 1, you explored the Zillow dataset, cleaned the data, and developed hypotheses about how preprocessing and feature engineering might improve predictive performance.

In this milestone, you will  develop, evaluate, and refine several machine learning models using those ideas. Rather than simply searching for the best algorithm, you will follow an iterative modeling workflow by:

1. Establishing baseline performance using several regression models.
2. Testing the preprocessing and feature engineering ideas proposed in Milestone 1.
3. Refining the feature set through feature selection.
4. Optimizing model performance through hyperparameter tuning.
5. Comparing the evolution of your models and selecting a final model to evaluate on the held-out test set.

Throughout this milestone, use **repeated 5-fold cross-validation (5 repeats)** to guide your modeling decisions. The held-out test set should be used only once, after all modeling decisions have been completed.




In [14]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, HistGradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))

## Prelude: Load Your Preprocessed Dataset from Milestone 1

In Milestone 1, you cleaned the Zillow dataset by removing unsuitable features, handling missing values, and encoding categorical variables. In this milestone, you will build, compare, and improve several regression models using that prepared dataset.

Begin by returning to your Milestone 1 notebook and rerunning your code through Part 3, where your dataset has been completely cleaned and encoded, but before any experimental feature engineering ideas were evaluated. Save this dataset and use it as the starting point for this milestone.

For example:

```python
# In Milestone 1
df_cleaned.to_csv("zillow_cleaned.csv", index=False)
```

```python
# In Milestone 2
df = pd.read_csv("zillow_cleaned.csv")
```

Next:

1. Separate the predictors (`X`) from the target (`y`).
2. Split the dataset into training and test sets using `train_test_split`.

Some regression models, such as **Ridge Regression** and **Lasso Regression**, require feature scaling. If you use one of these models, standardize the predictor variables **using only the training data**, then apply the same transformation to the test data.

```python
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
```

**Notes**

- Ordinary Linear Regression, Decision Trees, Random Forests, and HistGradientBoosting do **not** require feature scaling.
- If you create additional features later in this milestone and are using a scaled model, repeat the scaling step so the new features are transformed consistently.
- Throughout this milestone, use the same training/test split so that all models are evaluated on identical data.

In [15]:
# PRELUDE: Load and Prepare Milestone 1 Dataset

url = "https://www.cs.bu.edu/fac/snyder/cs505/Data/zillow_dataset.csv"

filename = os.path.basename(urlparse(url).path)

if not os.path.exists(filename):
    try:
        print("Downloading the file...")
        response = requests.get(url)
        response.raise_for_status()
        with open(filename, "wb") as f:
            f.write(response.content)
        print("File downloaded successfully.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
else:
    print("File already exists. Skipping download.")

df = pd.read_csv(filename)

File already exists. Skipping download.


 We dropped propertyzoningdesc column in the dataset: it has 1,907 unique values, and 544 (28.5%) appear only once in the dataset. If a code shows up only once, there's no way to tell whether its value is a real pattern or just a coincidence, since there's nothing else to compare it to. This makes over a quarter of the feature's categories essentially unusable. It's also a raw zoning code (e.g. LAR1, LAR3) where much of the remaining variety is just density tiers within the same city, not distinct property types. One-hot encoding it alone added around 1,907 columns, pushing our total to 2,744 and causing a MemoryError (around 1.05 GB) when converting to a DataFrame. Dropping it brought the total to 1,096 columns (around 0.44 GB), resolving the error. Location is still captured through fips, regionidcity, regionidzip, and regionidneighborhood.

In [16]:
# showing the number of unique values in the 'propertyzoningdesc' column to explain why we are dropping it from the dataset
# new in milestone 2, not in milestone 1
print("Number of unique values in 'propertyzoningdesc':", df['propertyzoningdesc'].nunique())

value_counts_propertyzoningdesc = df['propertyzoningdesc'].value_counts()

print("Number of rows with unique 'propertyzoningdesc'values:", (value_counts_propertyzoningdesc == 1).sum())  # 544

Number of unique values in 'propertyzoningdesc': 1907
Number of rows with unique 'propertyzoningdesc'values: 544


 We also dropped regionidneighborhood and regionidzip before encoding, in addition to propertyzoningdesc. regionidneighborhood has 480 unique values and 60.1% missing — already our weakest location feature,  and alone produced 459 of our 1,096 encoded columns (42%). regionidzip has low missingness (0.1%) and genuine signal, but its 389 unique values added another 382 columns. Together these two features were the main driver of long training times (Ridge alone took over 10 minutes, HistGradientBoosting over 10 minutes). We retain location signal through fips (county) and regionidcity (city), which capture similar geographic information at much lower cardinality.

In [17]:
# Milestone 1 imputation and encoding steps are performed here to prepare the dataset for modeling below.

# 3A: drop unsuitable features
df_suitable = df.drop(columns=['parcelid', 'assessmentyear', 'rawcensustractandblock', 'censustractandblock'])

# 3B: drop features missing more than 70%
missing_pct = df_suitable.isnull().mean() * 100
threshold = 70.0
columns_to_drop = missing_pct[missing_pct > threshold].index
df_missing = df_suitable.drop(columns=columns_to_drop)

# 3C: remove problematic samples
df_cleaned = df_missing.dropna(subset=["taxvaluedollarcnt"])
df_cleaned.dropna(thresh=20, inplace=True)

# 3D: train test split
X = df_cleaned.drop(columns=["taxvaluedollarcnt"])
y = df_cleaned["taxvaluedollarcnt"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

# 3E: imputation
X_train_header = X_train.columns.to_list()
missing_cols = []
for col in X_train_header:
    if X_train[col].isna().sum() != 0:
        missing_cols += [col]

below_five = []
for col in missing_cols:
    per = (X_train[col].isna().sum() / len(X_train)) * 100
    if per < 5:
        below_five += [col]

X_train.dropna(subset=below_five, inplace=True)
X_test.dropna(subset=below_five, inplace=True)
y_train = y_train.loc[X_train.index]
y_test = y_test.loc[X_test.index]

X_train["garagecarcnt"] = X_train["garagecarcnt"].fillna(0)
X_train["garagetotalsqft"] = X_train["garagetotalsqft"].fillna(0)
X_test["garagecarcnt"] = X_test["garagecarcnt"].fillna(0)
X_test["garagetotalsqft"] = X_test["garagetotalsqft"].fillna(0)

mode_cols = ["unitcnt", "regionidneighborhood", "heatingorsystemtypeid", "buildingqualitytypeid", "airconditioningtypeid"]
mode_imputer = SimpleImputer(strategy="most_frequent")
mode_imputer.fit(X_train[mode_cols])
X_train[mode_cols] = mode_imputer.transform(X_train[mode_cols])
X_test[mode_cols] = mode_imputer.transform(X_test[mode_cols])

# drop propertyzoningdesc due to extreme cardinality (see discussion above)
X_train = X_train.drop(columns=['propertyzoningdesc'])
X_test = X_test.drop(columns=['propertyzoningdesc'])

# drop additional high-cardinality location columns due to runtime constraints (see discussion above)
X_train = X_train.drop(columns=['regionidneighborhood'])
X_test = X_test.drop(columns=['regionidneighborhood'])

# drop regionidzip due to runtime constraints (see discussion above)
X_train = X_train.drop(columns=['regionidzip'])
X_test = X_test.drop(columns=['regionidzip'])

# 3F: encoding, fit encoder using only training data
categorical_ordinal = ["buildingqualitytypeid"]
categorical_nominal = ["regionidcity", "regionidcounty",  
                        "airconditioningtypeid", "fips", "heatingorsystemtypeid",
                        "propertylandusetypeid", "propertycountylandusecode"]

# importing Column Transformer and OneHotEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("ordinal", OrdinalEncoder(), categorical_ordinal),
        ("nominal", OneHotEncoder(handle_unknown="ignore"), categorical_nominal),
    ],
    remainder="passthrough"
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# create as DataFrame for easier handling and to maintain column names
feature_names = preprocessor.get_feature_names_out()

X_train_encoded = pd.DataFrame(X_train_encoded.toarray(), columns=feature_names, index=X_train.index)
X_test_encoded = pd.DataFrame(X_test_encoded.toarray(), columns=feature_names, index=X_test.index)

print("X_train_encoded shape:", X_train_encoded.shape)
print("X_test_encoded shape:", X_test_encoded.shape)

print("Missing values X_train_encoded:", X_train_encoded.isnull().sum().sum())
print("Missing values X_test_encoded:", X_test_encoded.isnull().sum().sum())

print("Number of features in X_train_encoded:", X_train_encoded.shape[1])
print("Number of features in X_test_encoded:", X_test_encoded.shape[1])
print("Shapes match:", X_train_encoded.shape[1] == X_test_encoded.shape[1])  # should be True

# scaling, fit on training data only, needed for Ridge Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_encoded shape: (51219, 255)
X_test_encoded shape: (12812, 255)
Missing values X_train_encoded: 0
Missing values X_test_encoded: 0
Number of features in X_train_encoded: 255
Number of features in X_test_encoded: 255
Shapes match: True
X_train_scaled shape: (51219, 255)
X_test_scaled shape: (12812, 255)


In [18]:
print("final X_train encoded columns:", X_train_encoded.columns.tolist())

final X_train encoded columns: ['ordinal__buildingqualitytypeid', 'nominal__regionidcity_3491.0', 'nominal__regionidcity_4406.0', 'nominal__regionidcity_5465.0', 'nominal__regionidcity_5534.0', 'nominal__regionidcity_6021.0', 'nominal__regionidcity_6395.0', 'nominal__regionidcity_6822.0', 'nominal__regionidcity_8384.0', 'nominal__regionidcity_9840.0', 'nominal__regionidcity_10241.0', 'nominal__regionidcity_10389.0', 'nominal__regionidcity_10608.0', 'nominal__regionidcity_10723.0', 'nominal__regionidcity_10734.0', 'nominal__regionidcity_10774.0', 'nominal__regionidcity_10815.0', 'nominal__regionidcity_11626.0', 'nominal__regionidcity_12292.0', 'nominal__regionidcity_12447.0', 'nominal__regionidcity_12520.0', 'nominal__regionidcity_12773.0', 'nominal__regionidcity_13091.0', 'nominal__regionidcity_13150.0', 'nominal__regionidcity_13232.0', 'nominal__regionidcity_13311.0', 'nominal__regionidcity_13693.0', 'nominal__regionidcity_13716.0', 'nominal__regionidcity_14111.0', 'nominal__regionidc

## Problem 1: Model Selection and Baselines [6 pts]

### 1.A Coding

Select **three** regression models from the following list and evaluate each one using the cleaned training dataset.

Use the default hyperparameters provided by scikit-learn (except where scaling is required).

Available models:

* Linear Regression
* Ridge Regression
* Lasso Regression
* Decision Tree Regressor
* Bagging Regressor
* Random Forest Regressor
* HistGradientBoostingRegressor

For each of the three models you choose:

* Train using the **training dataset only**.
* Use **Repeated 5-Fold Cross-Validation** (5 repeats).
* Report validation performance:

  * Mean CV MAE
  * Standard Deviation of CV MAE

In [19]:
# Add as many code cells as needed.
from sklearn.metrics import mean_absolute_error

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

results_log = []  
# accumulates every model result across the whole notebook

def evaluate_model(model, X, y, model_name, stage, cv=cv):
    """
    Fits a model, computes train MAE and repeated-CV MAE, times it,
    and logs the result for later comparison across all parts.
    """
    start = time.time()
    model.fit(X, y)
    train_mae = mean_absolute_error(y, model.predict(X))
    cv_scores = -cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error')
    elapsed = time.time() - start

    result = {
        "model": model_name,
        "stage": stage,
        "train_mae": train_mae,
        "cv_mae_mean": cv_scores.mean(),
        "cv_mae_std": cv_scores.std(),
        "time_seconds": elapsed
    }
    results_log.append(result)
    print(f"{stage} {model_name}: train MAE={train_mae:.0f}, CV MAE={cv_scores.mean():.0f}, "
          f"CV MAE Standard Deviation={cv_scores.std():.0f}, time={elapsed:.1f}s")
    return result

In [20]:
# run the model evaluation function for each model on the training data

# Three models selected for evaluation are Ridge Regression, Decision Tree, and HistGradientBoostingRegressor.
# The reason for selecting these models is to compare a linear model (Ridge Regression) with 
# two tree-based models (Decision Tree and HistGradientBoostingRegressor) to see how they perform on the dataset.
# Also, these 3 models are comparatively less complex and faster to train than some other models,
# making them suitable for initial evaluation.

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

evaluate_model(Ridge(), X_train_scaled, y_train, "Ridge", "Part 1 Baseline")

evaluate_model(DecisionTreeRegressor(random_state=random_state), X_train_encoded, y_train, "Decision Tree", "Part 1 Baseline")

evaluate_model(HistGradientBoostingRegressor(random_state=random_state), X_train_encoded, y_train, "HistGradientBoosting", "Part 1 Baseline");

Part 1 Baseline Ridge: train MAE=234516, CV MAE=235926, CV MAE Standard Deviation=2601, time=4.6s
Part 1 Baseline Decision Tree: train MAE=2108, CV MAE=262102, CV MAE Standard Deviation=3317, time=35.5s
Part 1 Baseline HistGradientBoosting: train MAE=186689, CV MAE=200236, CV MAE Standard Deviation=2895, time=164.0s


### 1.B Discussion

Answer the following questions.

#### 1.B.1

Which of your three models achieved the **lowest validation MAE score**?

> We chose Ridge, Decision Tree, and HistGradientBoosting to cover three different approaches: a regularized linear model, a single tree, and a boosted ensemble. Based on the model performance results above, HistGradientBoosting had the lowest CV MAE ($200,236), beating Ridge ($235,926) and Decision Tree ($262,102). 

#### 1.B.2

Which model produced the **smallest standard deviation** across the repeated cross-validation runs? What does this suggest about its stability?

> Ridge produced the smallest standard deviation (std=$2,601), compared to HistGradientBoosting (std=$2,895) and Decision Tree (std=$3,317). This suggests Ridge is the most stable of the three, and its performance stays consistent regardless of which properties land in training vs. validation, likely because its regularization keeps extreme outlier values from swinging the fit much from fold to fold.

#### 1.B.3

Did any model appear to overfit or underfit? Explain your reasoning using the training and cross-validation results.

> Comparing train MAE to CV MAE tells us whether a model is overfitting: if train MAE is much lower than CV MAE, the model fits the training data closely but fails to generalize to unseen data (overfitting). If both are high and similar, the model isn't capturing the pattern well at all (underfitting). Decision Tree appears to overfit strongly: train MAE ($2,108) is about 100 times smaller than CV MAE ($262,102), fitting a tree's typical behavior of memorizing individual properties, including outliers, instead of learning a general pattern. Ridge shows no real overfitting (train $234,516 vs CV $235,926) due to regularization. HistGradientBoosting shows mild overfitting (train $186,689 vs CV $200,236), likely because boosting focuses on cases like outliers, but the gap is much smaller than Decision Tree's.

#### 1.B.4

Compare the overall strengths and weaknesses of the three models. Did any model consistently perform better, or were there important tradeoffs between accuracy and stability?

> No single model was best across both accuracy and stability, and there's a real tradeoff between them. HistGradientBoosting had the lowest (best) CV MAE at $200,236, while Decision Tree had the highest (worst) at $262,102, a difference of over $60,000. On stability, Ridge had the lowest (best) standard deviation at $2,601, while Decision Tree again had the highest (worst) at $3,317. HistGradientBoosting is best on accuracy but not quite the most stable. Ridge trades has better stability but worse accuracy. Decision Tree is the weakest on both measures simultaneously.

## Part 2: Evaluate Your Feature Engineering Hypotheses [6 pts]

### 2.A Coding

In **Milestone 1**, you proposed several preprocessing and feature engineering ideas that you believed might improve predictive performance.

Select **at least three** of those ideas and evaluate them.

These may include, for example:

* Creating new features
* Transforming existing features
* Removing features
* Combining features
* Other preprocessing ideas that you proposed in Milestone 1

For each idea:

* Apply the preprocessing or feature engineering to the **training dataset only**.
* Retrain the same three baseline models from **Problem 1** using repeated 5-fold cross-validation (5 repeats). 
* Compare the validation performance (mean CV MAE) and stability (standard deviation of CV MAE) with your original baseline results


> One of the most important things you can learn is that **not every clever idea results in an improvement**--they have to be evaluated by careful experiment.  And negative results are valuable if they are carefully evaluated and discussed!

In [21]:
from sklearn.metrics import make_scorer

models_p2 = {
    "Ridge": Ridge(),
    "Decision Tree": DecisionTreeRegressor(random_state=random_state),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=random_state)
}

# Idea 1: Remove redundant features 
drop_cols = ['remainder__calculatedbathnbr', 'remainder__fullbathcnt', 'remainder__finishedsquarefeet12']
X_train_idea1 = X_train_encoded.drop(columns=drop_cols)

for name, model in models_p2.items():
    if name == "Ridge":
        X_data = StandardScaler().fit_transform(X_train_idea1)
    else:
        X_data = X_train_idea1
    evaluate_model(model, X_data, y_train, name, "Part 2 - Redundant Removed")

#  Idea 2: Log transform target and skewed features
log_feature_cols = ['remainder__calculatedfinishedsquarefeet', 'remainder__lotsizesquarefeet']
X_train_idea2 = X_train_encoded.copy()
for col in log_feature_cols:
    X_train_idea2[col] = np.log1p(X_train_idea2[col])
y_train_log = np.log1p(y_train)

# y_true_log is the actual real target value (log-form) for the held-out rows, 
# and y_pred_log is the model's prediction (also log-form) for those same rows,
# and cross_val_score's scoring= parameter 
# is what tells it to hand both of these to our custom function, instead of using its default scoring method
def log_mae_scorer(y_true_log, y_pred_log):
    y_true_actual = np.expm1(y_true_log)
    y_pred_actual = np.expm1(y_pred_log)
    return mean_absolute_error(y_true_actual, y_pred_actual)

custom_scorer = make_scorer(log_mae_scorer, greater_is_better=False)

for name, model in models_p2.items():
    if name == "Ridge":
        X_data = StandardScaler().fit_transform(X_train_idea2)
    else:
        X_data = X_train_idea2

    start = time.time()

    # fit once on full training data to get train MAE (back-transformed to dollars)
    model.fit(X_data, y_train_log)
    train_pred_log = model.predict(X_data)
    train_pred_actual = np.expm1(train_pred_log)
    train_mae = mean_absolute_error(y_train, train_pred_actual)

    # repeated CV for validation MAE, using our custom back-transforming scorer
    scores = cross_val_score(model, X_data, y_train_log, cv=cv, scoring=custom_scorer)
    mae_scores = -scores

    elapsed = time.time() - start

    print(f"Part 2 - Log Transform {name}: train MAE={train_mae:.2f}, "
          f"CV MAE mean={mae_scores.mean():.2f}, CV MAE std={mae_scores.std():.2f}, time={elapsed:.1f}s")

# Idea 3: Scaling comparison
for name, model in models_p2.items():
    X_scaled = StandardScaler().fit_transform(X_train_encoded)
    evaluate_model(model, X_scaled, y_train, name, "Part 2 - Scaled")

Part 2 - Redundant Removed Ridge: train MAE=234476, CV MAE=235880, CV MAE Standard Deviation=2612, time=11.1s
Part 2 - Redundant Removed Decision Tree: train MAE=2108, CV MAE=262427, CV MAE Standard Deviation=3793, time=61.6s
Part 2 - Redundant Removed HistGradientBoosting: train MAE=184634, CV MAE=200279, CV MAE Standard Deviation=2493, time=270.5s
Part 2 - Log Transform Ridge: train MAE=212439.42, CV MAE mean=213939.49, CV MAE std=5152.58, time=13.0s
Part 2 - Log Transform Decision Tree: train MAE=2139.85, CV MAE mean=265136.38, CV MAE std=3613.26, time=51.5s
Part 2 - Log Transform HistGradientBoosting: train MAE=194197.44, CV MAE mean=201846.25, CV MAE std=3500.58, time=367.2s
Part 2 - Scaled Ridge: train MAE=234516, CV MAE=235926, CV MAE Standard Deviation=2601, time=8.4s
Part 2 - Scaled Decision Tree: train MAE=2107, CV MAE=262525, CV MAE Standard Deviation=3505, time=78.5s
Part 2 - Scaled HistGradientBoosting: train MAE=186689, CV MAE=200236, CV MAE Standard Deviation=2895, time=

### 2.B Discussion

Answer the following questions.

#### 2.B.1

Which of your feature engineering ideas produced the largest improvement in validation performance?

> Part 1 baseline results, for reference: Ridge: train MAE=$234,516, CV MAE=$235,926, CV MAE std=$2,601; Decision Tree: train MAE=$2,108, CV MAE=$262,102, CV MAE std=$3,317; HistGradientBoosting: train MAE=$186,689, CV MAE=$200,236, CV MAE std=$2,895. In part 2, we tested three ideas from Milestone 1: removing redundant features, log-transforming the target and skewed features, and scaling. Compared to the Part 1 baseline, the log transform produced the largest improvement, but only for Ridge: CV MAE dropped from $235,926 to $213,939, roughly a 9% improvement. Neither redundant feature removal nor scaling produced a meaningful improvement for any model relative to their respective baselines.

#### 2.B.2

Were any of your ideas unsuccessful or did they reduce model performance? Briefly explain.

> Redundant feature removal and scaling had essentially no effect on any model, as both train and CV MAE stayed roughly the same across the board, which makes sense since the dropped columns were duplicates and scaling doesn't change tree-based models or meaningfully change Ridge here. The log transform showed mixed results. For Decision Tree, both train and CV MAE got slightly worse, and its already-severe overfitting got a bit worse too. For HistGradientBoosting, CV MAE dipped very slightly, but the gap between train and CV MAE shrank noticeably, which is a real improvement in overfitting, even though raw accuracy barely moved. 

#### 2.B.3

Did some models benefit more from feature engineering than others? If so, why do you think this occurred?

> Yes, ridge was the only model that benefited from feature engineering, specifically from the log transform. This makes sense given how each model works: Ridge fits a straight-line relationship between features and target, so reducing the target's skew directly helps it find a better-fitting line. Decision Tree and HistGradientBoosting instead split the data based on thresholds in the raw feature values, so they don't rely on the target's distribution shape the same way, and the log transform provided no benefit and even slightly hurt their performance.

#### 2.B.4

Which preprocessing or feature engineering changes will you keep for the remainder of the milestone? Briefly justify your decision.

> Idea 1 (redundant feature removal): We will keep this for all three models, since it simplifies the feature set without any measurable downside to train or CV MAE. Idea 2 (log transform): We will keep this for Ridge, since it clearly improved both train and CV MAE. We will also keep it for HistGradientBoosting, since although CV MAE was marginally worse, the meaningfully reduced overfitting gap suggests better generalization potential, which we consider more valuable heading into Part 4's tuning. We will not apply it for Decision Tree, since it worsened both CV MAE and overfitting with no offsetting benefit. Idea 3 (scaling): Ridge will continue to be scaled, as required for the model to function correctly, though our test confirmed this doesn't meaningfully change its performance. Decision Tree and HistGradientBoosting will remain unscaled, since scaling has no effect on tree-based models.

## Part 3: Refine the Feature Set [6 pts]

### 3.A Coding

Using your dataset after completing **Part 2** (including any preprocessing and feature engineering changes you decided to keep):

Investigate whether **feature selection** can further improve model performance.

You may use one or more of the following methods:

* Forward Selection (for linear regression models)
* Backward Selection (for linear regression models)
* Feature importance from tree-based models (for decision trees, Random Forests, Bagging, and HistGradientBoosting)
* Another reasonable feature selection method

For each of your three models:

* Select a subset of features using an appropriate feature selection method.
* Retrain the model using only the selected features.
* Evaluate the model using the same repeated cross-validation procedure as before.
* Report the validation performance (the mean and standard deviation of the CV MAE).

> Not every model will necessarily benefit from feature selection. Choose methods that are appropriate for the models you selected. Negative results are valuable if they are carefully evaluated and discussed!

In [22]:
# Add as many code cells as needed.

### 3.B Discussion

#### 3.B.1

Did feature selection improve the validation performance of any of your models?

> Replace this text with your answer.

#### 3.B.2

Were there features that were consistently retained (or consistently removed) across multiple models?

> Replace this text with your answer.

#### 3.B.3

Were any of your engineered features selected as important? If so, what does this suggest about the hypotheses you developed in Milestone 1?

> Replace this text with your answer.

#### 3.B.4

After feature selection, did simpler models perform as well as—or better than—the models using the full feature set? Briefly discuss any tradeoffs you observed between model complexity and predictive performance.

> Replace this text with your answer.

> Your text here

## Part 4: Tune Your Models [8 pts]

### 4.A Coding

Using the three models developed in **Part 3** (including your final preprocessing, feature engineering, and feature selection decisions):

Investigate whether **hyperparameter tuning** can further improve model performance.

For each of your three models:

* Select one or more important hyperparameters to tune.
* Use one or more appropriate tuning methods. Consider first using validation curves (`sweep_parameter`) to identify a promising region or performance plateau, followed by a focused search using methods such as:

    * GridSearchCV
    * RandomizedSearchCV
    * Another reasonable hyperparameter search method

* Choose hyperparameter values based on the validation results. If several nearby values produce similar validation performance (a performance plateau), prefer **values near the beginning of the plateau,** since they often produce simpler models with nearly identical predictive performance.
* Retrain the model using those hyperparameters.
* Evaluate the tuned model using repeated 5-fold cross-validation (5 repeats). 
* Report the validation performance (**mean** and **standard deviation** of the CV MAE).


In [23]:
# Add as many code cells as needed.

### 4.B Discussion

Answer the following questions.

#### 4.B.1

Which hyperparameters had the greatest impact on model performance? Briefly explain.

> Replace this text with your answer.

#### 4.B.2

Did hyperparameter tuning substantially improve the performance of all three models, or only some of them?

> Replace this text with your answer.

#### 4.B.3

Which tuning method(s) did you use for each model? Briefly explain why you chose those methods.

> Replace this text with your answer.

#### 4.B.4

After tuning, how did the relative performance of your three models change? Did tuning affect which model appeared to perform best?

> Replace this text with your answer.

## Part 5: Final Model and Workflow Assessment [14 pts]

### 5.A Coding

Using the work completed in **Parts 1–4**:

Select your **best-performing model** and prepare your final modeling pipeline.

Your pipeline should include all preprocessing, feature engineering, feature selection, and hyperparameter tuning decisions that you chose to retain.

Evaluate your final model by:

* Training on the complete training dataset.
* Reporting the **mean** and **standard deviation** of the repeated cross-validation MAE.
* Evaluating the model on the held-out test set.
* Reporting the final test MAE.

In [24]:
# Add as many code cells as needed.

### 5.B Discussion

Answer the following questions.

#### 5.B.1

Compare the performance of your final model with its original baseline from **Part 1**. Which changes contributed the most to the improvement?

> Replace this text with your answer.

#### 5.B.2

Looking back at the hypotheses you proposed in **Milestone 1**, which were supported by your experimental results? Were any hypotheses disproved?

> Replace this text with your answer.

#### 5.B.3

Why did you select this model as your final model? Discuss both its predictive performance and any other considerations (such as stability, simplicity, or interpretability).

> Replace this text with your answer.

#### 5.B.4

What did you learn about your dataset and the machine learning process through this end-to-end modeling workflow? If you had additional time, what would you investigate next?

> Replace this text with your answer.